# Enformer - regulatory effect prediction

Predicts the regulatory impact of the candidate variants from the VEP/CADD annotation notebook with **Enformer** (DeepMind, 2021), comparing REF and ALT sequences.

This notebook runs in Google Colab with a GPU runtime (T4); upload the annotation table from 7a before running it.

## Install dependencies and check GPU

In [ ]:
# Install dependencies
!pip install enformer-pytorch --quiet
!pip install einops --quiet

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU - switch the runtime to a T4 GPU before continuing')

## Setup, imports and configuration

In [ ]:
import io
import time
import traceback

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub

# Input: annotation table from 7a, uploaded to the Colab session (Files panel)
# Outputs are written to the same folder; download them at the end
ANC, DS     = 'EUR', 'WGS'
DIR_COLAB   = '/content'
ANNOT_FILE  = f'{DIR_COLAB}/VEP_CADD_annotation_{ANC}_{DS}.tsv'
SCORES_FILE = f'{DIR_COLAB}/enformer_scores_{ANC}_{DS}.tsv'
DELTA_FIG   = f'{DIR_COLAB}/enformer_delta_scores_{ANC}_{DS}.png'
HEATMAP_FIG = f'{DIR_COLAB}/enformer_heatmap_{ANC}_{DS}.png'

# Enformer (TF Hub) needs a 393,216 bp input window
ENFORMER_URL = 'https://tfhub.dev/deepmind/enformer/1'
SEQ_LENGTH   = 393216
BASES_MAP    = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

# Track metadata (Basenji/Enformer targets)
TRACKS_URL = ('https://raw.githubusercontent.com/calico/basenji/master'
              '/manuscripts/cross2020/targets_human.txt')

# Track groups used to summarise the predictions
KEYWORDS_INTEREST = [
    'H3K27ac',     # active enhancer
    'H3K4me1',     # poised enhancer
    'H3K4me3',     # promoter
    'DNASE',       # chromatin accessibility
    'ATAC',        # chromatin accessibility (alternative)
    'brain',       # brain tissue
    'neuro',       # neuronal tissue
    'CTCF',        # insulator
    'H3K9me3',     # heterochromatin / silencing
    'H3K36me3',    # active transcription
]
KEY_TRACKS = ['H3K27ac', 'DNASE', 'H3K4me1']   # heatmap
TOP_N = 15                                      # tracks shown per variant

## Load candidate variants

In [ ]:
annot = pd.read_csv(ANNOT_FILE, sep='\t')

VARIANTS = [
    {'chrom': str(r.CHROM), 'pos': int(r.POS), 'ref': str(r.REF), 'alt': str(r.ALT),
     'HAR': r.HAR, 'OR': r.OR, 'P': r.P, 'sig_tier': r.sig_tier,
     'rsID': r.rsID, 'genes': r.nearest_genes}
    for r in annot.itertuples()
]

print(f'{len(VARIANTS)} variants loaded from {ANNOT_FILE}')
for v in VARIANTS:
    print(f"  {v['HAR']} chr{v['chrom']}:{v['pos']} {v['ref']}>{v['alt']}  OR={v['OR']:.3f}")

## Load Enformer

In [ ]:
print('Loading Enformer...')
_hub_module   = hub.load(ENFORMER_URL)
_enformer_obj = _hub_module.model

# Work with the concrete function (predict_on_batch returns None in recent TF versions)
_input_spec  = tf.TensorSpec(shape=[None, SEQ_LENGTH, 4], dtype=tf.float32)
_concrete_fn = _enformer_obj.predict_on_batch.get_concrete_function(_input_spec)

print('Output keys:', list(_concrete_fn.structured_outputs.keys()))
print('Enformer ready.')

## Load track metadata

In [ ]:
resp = requests.get(TRACKS_URL, timeout=30)
resp.raise_for_status()

tracks_df = pd.read_csv(io.StringIO(resp.text), sep='\t').reset_index(drop=True)
print(f'Tracks loaded: {len(tracks_df)}')
tracks_df.head(3)

## Helper functions

In [ ]:
def one_hot_encode(seq):
    enc = np.zeros((len(seq), 4), dtype=np.float32)
    for i, b in enumerate(seq.upper()):
        idx = BASES_MAP.get(b, -1)
        if idx >= 0: enc[i, idx] = 1.0
    return enc

def fetch_sequence(chrom, center_pos_1based, length=SEQ_LENGTH):
    half  = length // 2
    start = max(0, center_pos_1based - 1 - half)
    end   = start + length
    url   = (f'https://api.genome.ucsc.edu/getData/sequence'
             f'?genome=hg38;chrom=chr{chrom};start={start};end={end}')
    resp  = requests.get(url, timeout=120)
    resp.raise_for_status()
    seq = resp.json()['dna'].upper()
    if len(seq) < length:
        seq = seq + 'N' * (length - len(seq))
    return seq, start

def make_alt_sequence(ref_seq, var_idx, ref_allele, alt_allele, length=SEQ_LENGTH):
    alt = ref_seq[:var_idx] + alt_allele + ref_seq[var_idx + len(ref_allele):]
    if   len(alt) < length: alt = alt + 'N' * (length - len(alt))
    elif len(alt) > length:
        trim = (len(alt) - length) // 2
        alt  = alt[trim: trim + length]
    return alt

def predict(seq_array):
    seq_tf = tf.constant(seq_array[np.newaxis], dtype=tf.float32)
    out    = _concrete_fn(seq_tf)
    key    = 'human' if 'human' in out else list(out.keys())[0]
    return out[key].numpy()[0]                  # (896, 5313)

def score_variant(v, n_center_bins=11):
    chrom, pos, ref, alt = v['chrom'], v['pos'], v['ref'], v['alt']
    label = f"{v['HAR']} chr{chrom}:{pos} {ref}>{alt}"
    print(f'\n{label}')

    seq, seq_start = fetch_sequence(chrom, pos)
    var_idx    = pos - 1 - seq_start
    actual_ref = seq[var_idx: var_idx + len(ref)]
    if actual_ref != ref:
        print(f'  REF mismatch: expected={ref}, got={actual_ref}')

    ref_enc = one_hot_encode(seq)
    alt_enc = one_hot_encode(make_alt_sequence(seq, var_idx, ref, alt))

    print('  Enformer REF...')
    pred_ref = predict(ref_enc)
    print('  Enformer ALT...')
    pred_alt = predict(alt_enc)

    center     = pred_ref.shape[0] // 2
    half_b     = n_center_bins // 2
    sl         = slice(center - half_b, center + half_b + 1)
    eps        = 1e-6
    delta_mean = np.log2((pred_alt[sl] + eps) / (pred_ref[sl] + eps)).mean(axis=0)

    print('  done')
    return {**v, 'label': label, 'pred_ref': pred_ref,
            'pred_alt': pred_alt, 'delta_mean': delta_mean}

## Run Enformer on the variants

In [ ]:
results = []
for v in VARIANTS:
    try:
        results.append(score_variant(v))
        time.sleep(1)
    except Exception:
        print(f"  ERROR in {v['HAR']} chr{v['chrom']}:{v['pos']}:")
        traceback.print_exc()

print(f'\n{len(results)}/{len(VARIANTS)} variants processed')

## Select tracks of interest

In [ ]:
mask = tracks_df['description'].str.lower().str.contains(
    '|'.join(kw.lower() for kw in KEYWORDS_INTEREST)
)
tracks_interest = tracks_df[mask]
idx_interest    = tracks_interest.index.tolist()
print(f'Tracks of interest: {len(idx_interest)}')
tracks_interest.head(20)

## Summary table: most affected tracks per variant

In [ ]:
summary_rows = []
for r in results:
    delta = r['delta_mean']   # (5313,)
    delta_interest = sorted(((idx, delta[idx]) for idx in idx_interest),
                            key=lambda x: abs(x[1]), reverse=True)

    for rank, (idx, d) in enumerate(delta_interest[:TOP_N], start=1):
        summary_rows.append({
            'HAR':        r['HAR'],
            'variant':    f"{r['ref']}>{r['alt']}",
            'pos':        r['pos'],
            'OR':         r['OR'],
            'sig_tier':   r['sig_tier'],
            'track_idx':  idx,
            'track_desc': tracks_df.loc[idx, 'description'],
            'log2FC':     round(d, 4),
            'rank':       rank,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.head(30)

## Delta scores per variant

In [ ]:
fig, axes = plt.subplots(len(results), 1,
                         figsize=(14, 3 * len(results)),
                         constrained_layout=True, squeeze=False)

COL_UP, COL_DOWN = '#C62828', '#1565C0'   # signal increase / decrease

for ax, r in zip(axes[:, 0], results):
    delta = r['delta_mean']
    delta_top = sorted(((idx, delta[idx]) for idx in idx_interest),
                       key=lambda x: abs(x[1]), reverse=True)[:TOP_N]

    labels = [tracks_df.loc[i, 'description'][:50] for i, _ in delta_top]
    values = [d for _, d in delta_top]
    colors = [COL_UP if v > 0 else COL_DOWN for v in values]

    ax.barh(range(len(values)), values, color=colors, alpha=0.85)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('log₂FC (ALT/REF)', fontsize=8)
    ax.set_title(f"{r['HAR']}  chr{r['chrom']}:{r['pos']} {r['ref']}>{r['alt']}  "
                 f"OR={r['OR']:.3f}  [{r['sig_tier']}]", fontsize=8, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle(f'Enformer - top affected regulatory tracks (log₂FC ALT vs REF) - {ANC} {DS}',
             fontsize=10, fontweight='bold')
plt.savefig(DELTA_FIG, dpi=150, bbox_inches='tight')
plt.show()

## Heatmap: variants x H3K27ac / DNASE / H3K4me1 tracks

In [ ]:
key_mask = tracks_df['description'].str.lower().str.contains(
    '|'.join(k.lower() for k in KEY_TRACKS)
)
key_idx = tracks_df[key_mask].index.tolist()[:50]

matrix     = np.array([r['delta_mean'][key_idx] for r in results])
row_labels = [f"{r['HAR']} {r['ref']}>{r['alt']} [{r['sig_tier']}]" for r in results]
col_labels = [tracks_df.loc[i, 'description'][:35] for i in key_idx]

vmax = np.percentile(np.abs(matrix), 95)

fig, ax = plt.subplots(figsize=(18, max(4, 0.5 * len(results))))
im = ax.imshow(matrix, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=90, fontsize=6)
plt.colorbar(im, ax=ax, label='log₂FC (ALT/REF)', shrink=0.8)
ax.set_title('Enformer - log₂FC per variant x H3K27ac / DNASE / H3K4me1 tracks\n'
             'Red = ALT increases signal | Blue = ALT decreases signal', fontsize=9)
plt.tight_layout()
plt.savefig(HEATMAP_FIG, dpi=150, bbox_inches='tight')
plt.show()

## Save results

In [ ]:
summary_df.to_csv(SCORES_FILE, sep='\t', index=False)
print(f'Scores saved: {SCORES_FILE}')
print(f'Figures saved: {DELTA_FIG}, {HEATMAP_FIG}')